# Web Scrapping: Selenium

In [ ]:
pip install selenium

## 1. Libraries

In [3]:
import pandas as pd
import time

# Herramientas de Selenium
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException


## 2.  Configuración e Inicialización del Navegador

Aquí configuramos y lanzamos el navegador Chrome que será controlado por nuestro script. Dejamos que el Selenium Manager integrado se encargue de gestionar el chromedriver por nosotros, lo que simplifica mucho la configuración.

In [7]:
# Configuramos las opciones de Chrome
chrome_options = Options()
# chrome_options.add_argument("--headless")
chrome_options.add_argument("--start-maximized")
chrome_options.add_argument("--lang=en-US")

# Iniciar el WebDriver de Chrome
# Pista: Crea una variable llamada 'driver' y asígnale la instancia de webdriver.Chrome(),
# pasando nuestras 'chrome_options' como argumento.
driver = webdriver.Chrome(options=chrome_options)
print("WebDriver iniciado con éxito.")

SessionNotCreatedException: Message: session not created: probably user data directory is already in use, please specify a unique value for --user-data-dir argument, or don't use --user-data-dir; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#sessionnotcreatedexception
Stacktrace:
#0 0x56d12544a02a <unknown>
#1 0x56d124ee9a70 <unknown>
#2 0x56d124f24e07 <unknown>
#3 0x56d124f1f5a7 <unknown>
#4 0x56d124f6f93e <unknown>
#5 0x56d124f6ef06 <unknown>
#6 0x56d124f611b3 <unknown>
#7 0x56d124f2d59b <unknown>
#8 0x56d124f2e971 <unknown>
#9 0x56d12540f1fb <unknown>
#10 0x56d125412f49 <unknown>
#11 0x56d1253f62d9 <unknown>
#12 0x56d125413af8 <unknown>
#13 0x56d1253dabbf <unknown>
#14 0x56d1254370b8 <unknown>
#15 0x56d125437296 <unknown>
#16 0x56d125449006 <unknown>
#17 0x7c355b60eac3 <unknown>


## 3. Navegar a la Página de IMdb

Navegamos a la URL del Top 250 de IMDb. El paso más importante aquí es usar WebDriverWait. Le decimos a Selenium que espere hasta 10 segundos a que la lista de películas sea visible en la página antes de intentar hacer cualquier cosa. Esto hace nuestro script robusto frente a conexiones lentas.

In [ ]:
# URL del Top 250 de IMDb
url = "https://www.imdb.com/chart/top/"

# Pista: El objeto 'driver' tiene un método para abrir una URL. ¿Cuál es?
driver.get(url)

# Lista para guardar los datos de cada película
movies_data = []

# Selector CSS para la lista que contiene todas las películas
movie_list_selector = "ul.ipc-metadata-list"

try:
    print("Esperando a que la lista de películas cargue...")

    # Completa la espera para que el script se detenga hasta que la lista de películas sea visible.
    # Pista: Usa EC.visibility_of_element_located() y pásale una tupla con el método de búsqueda (By) y el selector.
    WebDriverWait(driver, 10).until(
        EC.visibility_of_element_located(( By.CSS_SELECTOR, movie_list_selector ))
    )
    print("Lista de películas encontrada. ¡A scrapear!")

except TimeoutException:
    print("Error: La lista de películas no cargó a tiempo. El script se detendrá.")
    driver.quit()

Esperando a que la lista de películas cargue...
Lista de películas encontrada. Comenzando el scraping.


## 4. Bucle Principal de Scraping

Este es el núcleo de nuestro scraper.

- Localizamos todos los elementos <*li> que contienen la información de cada película.
- Iteramos sobre los primeros 50 elementos de esa lista.
- Dentro de cada <*li>, buscamos los datos específicos (rango, título, año, calificación y URL) usando selectores CSS.
- Usamos bloques try-except para cada atributo. Si un dato no se encuentra en una película, el script registrará "No disponible" y continuará, en lugar de detenerse por un error.

In [ ]:
# Selector para cada item (película) en la lista
movie_item_selector = "li.ipc-metadata-list-summary-item"

# Pista: Usa el método 'find_elements' del driver para obtener una lista de todos los elementos que coincidan con 'movie_item_selector'.
movie_elements = driver.find_elements(By.CSS_SELECTOR, movie_item_selector)

# Iteramos solo sobre las primeras 50 películas
for movie in movie_elements[:50]:
    try:
        # --- Rango y Título ---
        # Pista: Primero, encuentra el elemento h3 con la clase 'ipc-title__text'. Luego, obtén su '.text'.
        title_element = movie.find_element(By.CSS_SELECTOR, "h3.ipc-title__text")
        full_title_text = title_element.text
        rank, title = full_title_text.split('. ', 1)

        # --- Año ---
        year = movie.find_element(By.CSS_SELECTOR, "div.cli-title-metadata > span").text

        # --- Calificación ---
        rating_element = movie.find_element(By.CSS_SELECTOR, "span.ipc-rating-star")
        rating = rating_element.text.split('\n')[0]

        # --- URL de la película ---
        # Pista: El enlace está en el atributo 'href' de la etiqueta <a>. Usa '.get_attribute()'
        url_element = movie.find_element(By.CSS_SELECTOR, "a.ipc-title-link-wrapper")
        movie_url = url_element.get_attribute('href')

        # Asegúrate de que los nombres de las variables coincidan con las que creaste arriba.
        movies_data.append({
            "Rango": rank,,
            "Titulo": title,
            "Año": year,
            "Calificacion_IMDb": rating,
            "URL": movie_url
        })
        print(f" Scraped: #{rank} {title}")

    except Exception as e:
        print(f" Error extrayendo datos de una película. Error: {e}")
        continue

print(f"\nScraping completado. Se extrajeron datos de {len(movies_data)} películas.")

✅ Scraped: #1 The Shawshank Redemption
✅ Scraped: #2 The Godfather
✅ Scraped: #3 The Dark Knight
✅ Scraped: #4 The Godfather Part II
✅ Scraped: #5 12 Angry Men
✅ Scraped: #6 The Lord of the Rings: The Return of the King
✅ Scraped: #7 Schindler's List
✅ Scraped: #8 Pulp Fiction
✅ Scraped: #9 The Lord of the Rings: The Fellowship of the Ring
✅ Scraped: #10 The Good, the Bad and the Ugly
✅ Scraped: #11 Forrest Gump
✅ Scraped: #12 The Lord of the Rings: The Two Towers
✅ Scraped: #13 Fight Club
✅ Scraped: #14 Inception
✅ Scraped: #15 Star Wars: Episode V - The Empire Strikes Back
✅ Scraped: #16 The Matrix
✅ Scraped: #17 Goodfellas
✅ Scraped: #18 Interstellar
✅ Scraped: #19 One Flew Over the Cuckoo's Nest
✅ Scraped: #20 Se7en
✅ Scraped: #21 It's a Wonderful Life
✅ Scraped: #22 The Silence of the Lambs
✅ Scraped: #23 Seven Samurai
✅ Scraped: #24 Saving Private Ryan
✅ Scraped: #25 The Green Mile
✅ Scraped: #26 City of God
✅ Scraped: #27 Life Is Beautiful
✅ Scraped: #28 Terminator 2: Judgment D

## 5. Crear el DataFrame y Guardar los Datos

Una vez que tenemos nuestra lista de diccionarios, la convertimos en un DataFrame de pandas, que es una estructura tipo tabla ideal para el análisis y almacenamiento de datos. Finalmente, lo guardamos en un archivo CSV y cerramos el navegador para liberar los recursos del sistema.

In [ ]:
if movies_data:
    # Pista: Llama a pd.DataFrame() y pásale la lista que contiene todos nuestros datos.
    df = pd.DataFrame(movies_data)
    # Pista: Usa el método '.to_csv()'. Dale un nombre de archivo, por ejemplo, "imdb_top_50.csv", y no te olvides de poner index=False.
    df.to_csv("imdb_top_50.csv", index=False)

    print("\n Datos guardados exitosamente.")
    display(df.head())
else:
    print("\nNo se pudo extraer ningún dato de las películas.")

# Cerrar el navegador
# Pista: Hay un método en el objeto 'driver' para cerrar todas las ventanas y terminar la sesión.
driver.quit()

print("\nNavegador cerrado correctamente. ¡Ejercicio terminado!")



🎉 Datos guardados exitosamente en 'imdb_top_50_peliculas.csv'


,Rango,Titulo,Año,Calificacion_IMDb,URL
0,1,The Shawshank Redemption,1994,9.3,https://www.imdb.com/title/tt0111161/?ref_=cht...
1,2,The Godfather,1972,9.2,https://www.imdb.com/title/tt0068646/?ref_=cht...
2,3,The Dark Knight,2008,9.1,https://www.imdb.com/title/tt0468569/?ref_=cht...
3,4,The Godfather Part II,1974,9.0,https://www.imdb.com/title/tt0071562/?ref_=cht...
4,5,12 Angry Men,1957,9.0,https://www.imdb.com/title/tt0050083/?ref_=cht...



Navegador cerrado correctamente.
